# 🚀 PEFT-SLM — Fully Self-Contained Colab Notebook
### M.Tech: Parameter-Efficient Fine-Tuning of Small Language Models for Document Analysis

---
**No uploads, no Google Drive, no repo needed.**  
Every source file is written by this notebook from scratch. Just run top to bottom.

| Step | What it does | Time |
|---|---|---|
| 0 — GPU check | Verify T4 is active | <1 min |
| 1 — Install | `pip install` all packages | ~3 min |
| 2 — Write source files | Creates all `src/`, `configs/` etc. in `/content/` | <1 min |
| 3 — Smoke test | Verifies every pipeline component (9 checks) | ~5 min |
| 4 — Quick demo | 3-way comparison on 5 built-in examples | ~10 min |
| 5A — Base SLM | Zero-shot baseline on SQuAD v2 (500 examples) | ~20 min |
| 5B — LoRA training | Fine-tune PEFT adapter ← your result | ~90 min |
| 5C — LLM baseline | Qwen1.5-7B zero-shot on same test set | ~25 min |
| 5D — Results table | Print 3-way comparison + charts | <1 min |
| 6A/6B — XL-Sum | Repeat for summarization task | ~2.5 hr |
| 7 — Save | Download results.csv to your PC | <1 min |

> ⚠️ **Before starting:** Go to `Runtime → Change runtime type → T4 GPU`

---
## 🖥️ Step 0 — GPU Check

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected!\n"
        "Fix: Runtime → Change runtime type → Hardware accelerator → T4 GPU\n"
        "Then: Runtime → Disconnect and delete runtime → reconnect."
    )

name   = torch.cuda.get_device_name(0)
vram   = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"✅ GPU  : {name}")
print(f"✅ VRAM : {vram:.1f} GB")
print(f"   PyTorch {torch.__version__} | CUDA {torch.version.cuda}")
if vram < 12:
    print("⚠️  < 12 GB VRAM — the 7B LLM baseline will need 4-bit quantization (handled automatically).")

---
## 📦 Step 1 — Install Dependencies

In [ ]:
# ── Install packages (takes ~3-5 min) ────────────────────────────────────────
# pymupdf / pytesseract are Phase 2 OCR tools — NOT needed for Phase 1.
# Removing them avoids  ImportError: cannot import name '_Ink' from 'PIL._typing'
# which happens when pymupdf forces Pillow >= 11 onto a Python 3.13 runtime.
# Pillow is pinned to <11 to prevent that conflict.
import subprocess, sys
pkgs = [
    "torch", "transformers", "datasets", "peft", "accelerate", "bitsandbytes",
    "evaluate", "rouge-score", "nltk",
    "Pillow>=10.0,<11.0",
    "pandas", "numpy", "pyyaml", "tqdm", "scikit-learn",
    "matplotlib", "seaborn", "tensorboard", "ipywidgets",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)
print("\n✅ All packages installed.")
print()
print("=" * 55)
print("REQUIRED: Runtime -> Restart session  (or Ctrl+M .)")
print("After restart, re-run THIS cell to verify, then Step 2.")
print("=" * 55)

---
## 🗂️ Step 2 — Write All Source Files
This cell creates the entire project structure in `/content/Mtech/` from scratch.

In [ ]:
import os, pathlib, inspect

ROOT = pathlib.Path("/content/Mtech")

def write(rel_path, content):
    """Write content to a file, stripping common leading whitespace."""
    p = ROOT / rel_path
    p.parent.mkdir(parents=True, exist_ok=True)
    # inspect.cleandoc trims the leading blank line AND the consistent indentation
    # that comes from embedding code inside an indented triple-quoted string.
    p.write_text(inspect.cleandoc(content) + "\n", encoding="utf-8")

# ── src/__init__.py ────────────────────────────────────────────────────────────
write("src/__init__.py", "")
write("src/data/__init__.py", "")
write("src/training/__init__.py", "")
write("src/evaluation/__init__.py", "")
write("src/utils/__init__.py", "")
write("src/extraction/__init__.py", "")
write("experiments/.gitkeep", "")
write("data/.gitkeep", "")

# ── src/utils/logging_setup.py ─────────────────────────────────────────────────
write("src/utils/logging_setup.py", """
    import logging, sys
    def setup_logging(level="INFO"):
        numeric = getattr(logging, level.upper(), logging.INFO)
        handler = logging.StreamHandler(sys.stdout)
        handler.setLevel(numeric)
        handler.setFormatter(logging.Formatter(
            fmt="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
            datefmt="%Y-%m-%d %H:%M:%S"))
        root = logging.getLogger()
        root.setLevel(numeric)
        root.handlers = [handler]
        for lib in ["urllib3","filelock","git","PIL"]:
            logging.getLogger(lib).setLevel(logging.WARNING)
""")

# ── src/utils/config_loader.py ─────────────────────────────────────────────────
write("src/utils/config_loader.py", """
    import yaml
    from pathlib import Path
    def load_config(path):
        p = Path(path)
        if not p.exists():
            raise FileNotFoundError(f"Config not found: {p}")
        with p.open("r", encoding="utf-8") as f:
            cfg = yaml.safe_load(f)
        required = ["task","model_name_or_path","dataset","training"]
        missing = [k for k in required if k not in cfg]
        if missing:
            raise ValueError(f"Config missing keys: {missing}")
        return cfg
""")

# ── src/utils/experiment_tracker.py ───────────────────────────────────────────
write("src/utils/experiment_tracker.py", """
    import csv, logging
    from pathlib import Path
    logger = logging.getLogger(__name__)
    _COLS = ["experiment","model","dataset","task","method","train_samples",
             "val_samples","test_samples","exact_match","f1","rouge1","rouge2",
             "rougeL","accuracy","macro_f1","train_time_s","peak_gpu_mb",
             "avg_latency_ms","trainable_pct","checkpoint"]
    class ExperimentTracker:
        def __init__(self, path="experiments/results.csv"):
            self.path = Path(path)
            self.path.parent.mkdir(parents=True, exist_ok=True)
        def log(self, row):
            existing = self._cols()
            new_keys = [k for k in row if k not in existing]
            all_cols = self._sort(existing + new_keys)
            exists = self.path.exists()
            if not exists or new_keys:
                self._rewrite_header(all_cols)
            with self.path.open("a", newline="", encoding="utf-8") as f:
                w = csv.DictWriter(f, fieldnames=all_cols, extrasaction="ignore")
                if not exists:
                    w.writeheader()
                w.writerow(row)
        def load(self):
            import pandas as pd
            if not self.path.exists():
                return pd.DataFrame()
            return pd.read_csv(self.path)
        def print_summary(self):
            df = self.load()
            if df.empty:
                print("No experiments logged yet."); return
            cols = [c for c in _COLS if c in df.columns]
            print(df[cols].to_string(index=False))
        def _cols(self):
            if not self.path.exists(): return []
            with self.path.open("r", encoding="utf-8") as f:
                r = csv.reader(f)
                try: return next(r)
                except StopIteration: return []
        def _sort(self, cols):
            pref = [c for c in _COLS if c in cols]
            ext  = [c for c in cols if c not in _COLS]
            return pref + ext
        def _rewrite_header(self, cols):
            if not self.path.exists(): return
            lines = self.path.read_text(encoding="utf-8").splitlines()
            if not lines: return
            lines[0] = ",".join(cols)
            self.path.write_text("\n".join(lines)+"\n", encoding="utf-8")
""")

# ── src/data/prompts.py ────────────────────────────────────────────────────────
write("src/data/prompts.py", """
    import re
    UNANSWERABLE = "unanswerable"
    QA_SEPARATOR = "### Answer:\\n"
    SUMMARY_SEPARATOR = "### Summary:\\n"
    _WS = re.compile(r"[ \\t\\r\\f\\v]+")
    _BL = re.compile(r"\\n{3,}")
    def clean_text(text):
        text = (text or "").replace("\\u00a0"," ").replace("\\u200b","")
        text = _WS.sub(" ", text)
        text = _BL.sub("\\n\\n", text)
        return "\\n".join(l.strip() for l in text.split("\\n")).strip()
    def build_qa_prompt(context, question):
        return (f"### Context:\\n{clean_text(context)}\\n\\n"
                f"### Question:\\n{clean_text(question)}\\n\\n"
                f"{QA_SEPARATOR}")
    def qa_completion(answers):
        texts = answers.get("text") or []
        return clean_text(texts[0]) if texts else UNANSWERABLE
    def qa_references(answers):
        texts = [clean_text(t) for t in (answers.get("text") or [])]
        return texts or [UNANSWERABLE]
    def build_summarization_prompt(article):
        return f"### Article:\\n{clean_text(article)}\\n\\n{SUMMARY_SEPARATOR}"
    def truncate_to_tokens(text, tokenizer, max_tokens):
        ids = tokenizer(text, add_special_tokens=False)["input_ids"]
        if len(ids) <= max_tokens: return text
        return tokenizer.decode(ids[:max_tokens], skip_special_tokens=True).rstrip()
    def build_training_text(prompt, completion, eos_token):
        return f"{prompt}{completion}{eos_token or ''}"
""")

# ── src/data/preprocessor.py ───────────────────────────────────────────────────
write("src/data/preprocessor.py", """
    import logging
    logger = logging.getLogger(__name__)
    _IGNORE_INDEX = -100
    _COMPLETION_SEPARATORS = [
        "### Answer:\\n", "### Summary:\\n",
        "### Category:\\n", "### Extracted fields (key: value):\\n",
    ]
    def tokenize_qa(examples, tokenizer, cfg):
        return _tokenize_prompt_completion(examples, tokenizer,
            cfg["dataset"]["max_source_length"]+cfg["dataset"]["max_target_length"])
    def tokenize_seq2seq(examples, tokenizer, cfg):
        return _tokenize_prompt_completion(examples, tokenizer,
            cfg["dataset"]["max_source_length"]+cfg["dataset"]["max_target_length"])
    def tokenize_classification(examples, tokenizer, cfg):
        return _tokenize_prompt_completion(examples, tokenizer,
            cfg["dataset"]["max_source_length"]+cfg["dataset"].get("max_target_length",16))
    def _tokenize_prompt_completion(examples, tokenizer, max_length):
        all_ids, all_masks, all_labels, skipped = [], [], [], 0
        prompts = examples["prompt"] if isinstance(examples["prompt"],list) else [examples["prompt"]]
        for prompt in prompts:
            full = tokenizer(prompt, max_length=max_length, truncation=True, padding=False, return_tensors=None)
            input_ids = full["input_ids"]; attn = full["attention_mask"]
            pp = _extract_prompt_part(prompt)
            prompt_len = len(tokenizer(pp, max_length=max_length, truncation=True, padding=False, return_tensors=None)["input_ids"])
            labels = list(input_ids)
            for i in range(min(prompt_len, len(labels))): labels[i] = _IGNORE_INDEX
            comp = [t for t in labels if t != _IGNORE_INDEX]
            if not comp:
                logger.warning("SKIPPING example — no completion tokens after masking."); skipped+=1; continue
            all_ids.append(input_ids); all_masks.append(attn); all_labels.append(labels)
        if skipped: logger.warning("%d/%d examples skipped.", skipped, len(prompts))
        return {"input_ids":all_ids,"attention_mask":all_masks,"labels":all_labels}
    def _extract_prompt_part(full_prompt):
        for sep in _COMPLETION_SEPARATORS:
            if sep in full_prompt:
                return full_prompt[:full_prompt.rfind(sep)+len(sep)]
        return full_prompt
    def get_data_collator(tokenizer):
        from transformers import DataCollatorForSeq2Seq
        return DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True,
            label_pad_token_id=_IGNORE_INDEX, pad_to_multiple_of=8)
""")

# ── src/data/dataset_loader.py ─────────────────────────────────────────────────
write("src/data/dataset_loader.py", """
    from __future__ import annotations
    import json, logging
    from pathlib import Path
    from .prompts import (UNANSWERABLE, build_qa_prompt, build_summarization_prompt,
                          build_training_text, clean_text, qa_completion, truncate_to_tokens)
    logger = logging.getLogger(__name__)
    _DATASET_ALIASES = {"squad_v2":"rajpurkar/squad_v2"}
    _XLSUM_REPO = "csebuetnlp/xlsum"
    _XLSUM_PARQUET = "hf://datasets/csebuetnlp/xlsum@refs%2Fconvert%2Fparquet"
    _PROMPT_OVERHEAD_TOKENS = 24
    _SQUAD_SPLIT_SOURCE = {"train":"train","validation":"validation","test":"validation"}
    _HF_SPLIT_SOURCE = {
        "squad_v2": _SQUAD_SPLIT_SOURCE,
        "rajpurkar/squad_v2": _SQUAD_SPLIT_SOURCE,
        "_default": {"train":"train","validation":"validation","test":"test"},
    }
    def load_dataset_for_task(cfg, tokenizer=None):
        task = cfg["task"]; ds_cfg = cfg["dataset"]
        if task == "qa":           return _load_qa(ds_cfg, tokenizer, cfg)
        if task == "summarization":return _load_summarization(ds_cfg, tokenizer, cfg)
        raise ValueError(f"Unknown task: {task}")
    def _load_raw(name, subset=None):
        from datasets import load_dataset
        name = _DATASET_ALIASES.get(name, name)
        if name == _XLSUM_REPO:
            lang = subset or "english"
            base = f"{_XLSUM_PARQUET}/{lang}"
            return load_dataset("parquet",
                data_files={s: f"{base}/{s}/*.parquet" for s in ("train","validation","test")})
        return load_dataset(name, subset) if subset else load_dataset(name)
    def _load_qa(ds_cfg, tokenizer, cfg):
        from .preprocessor import tokenize_qa
        name = ds_cfg.get("name","rajpurkar/squad_v2")
        raw = _load_raw(name, ds_cfg.get("subset"))
        raw = _subsample(raw, ds_cfg, cfg)
        if tokenizer is None: return raw
        max_src = ds_cfg["max_source_length"]
        def _fmt(ex):
            q_len = len(tokenizer(ex["question"],add_special_tokens=False)["input_ids"])
            budget = max(max_src - q_len - _PROMPT_OVERHEAD_TOKENS, 32)
            ctx = truncate_to_tokens(clean_text(ex["context"]), tokenizer, budget)
            ans = qa_completion(ex["answers"])
            keep = ans == UNANSWERABLE or ans.lower() in ctx.lower()
            return {"prompt": build_training_text(build_qa_prompt(ctx,ex["question"]),ans,tokenizer.eos_token),
                    "answer": ans, "keep": keep}
        fmt = raw.map(_fmt, remove_columns=raw["train"].column_names)
        n = len(fmt["train"])
        fmt["train"] = fmt["train"].filter(lambda e: e["keep"])
        if len(fmt["train"]) < n:
            logger.info("Dropped %d train examples (answer truncated).", n-len(fmt["train"]))
        fmt = fmt.remove_columns("keep")
        return fmt.map(lambda ex: tokenize_qa(ex,tokenizer,cfg), batched=True,
                        remove_columns=["prompt","answer"])
    def _load_summarization(ds_cfg, tokenizer, cfg):
        from .preprocessor import tokenize_seq2seq
        name = ds_cfg.get("name","csebuetnlp/xlsum")
        raw = _load_raw(name, ds_cfg.get("subset","english"))
        raw = _subsample(raw, ds_cfg, cfg)
        if tokenizer is None: return raw
        max_src = ds_cfg["max_source_length"]; max_tgt = ds_cfg["max_target_length"]
        def _fmt(ex):
            art = truncate_to_tokens(clean_text(ex.get("text","")), tokenizer, max_src-_PROMPT_OVERHEAD_TOKENS)
            summ = truncate_to_tokens(clean_text(ex.get("summary","")), tokenizer, max_tgt-1)
            return {"prompt": build_training_text(build_summarization_prompt(art),summ,tokenizer.eos_token),
                    "summary": summ}
        fmt = raw.map(_fmt, remove_columns=raw["train"].column_names)
        return fmt.map(lambda ex: tokenize_seq2seq(ex,tokenizer,cfg), batched=True,
                        remove_columns=["prompt","summary"])
    def _subsample(raw, ds_cfg, cfg):
        import numpy as np
        from datasets import DatasetDict
        seed = cfg.get("training",{}).get("seed",42)
        rng = np.random.default_rng(seed)
        dataset_name = ds_cfg.get("name","")
        srcs = _HF_SPLIT_SOURCE.get(dataset_name, _HF_SPLIT_SOURCE["_default"])
        n_tr = ds_cfg.get("train_samples"); n_va = ds_cfg.get("val_samples"); n_te = ds_cfg.get("test_samples")
        result = {}
        val_src = srcs["validation"]; test_src = srcs["test"]
        if val_src == test_src and val_src in raw:
            shared = raw[val_src]
            total = (n_va or 0)+(n_te or 0)
            if total > 0 and total < len(shared):
                idx = rng.choice(len(shared),size=total,replace=False).tolist(); idx.sort()
                vi = idx[:n_va]; ti = idx[n_va:]
            else:
                all_i = list(range(len(shared)))
                vi = all_i[:n_va] if n_va else all_i
                ti = all_i[n_va:n_va+(n_te or 0)] if n_te else []
            if vi: result["validation"] = shared.select(vi)
            if ti:
                result["test"] = shared.select(ti)
                _save_test_indices(ti, cfg, seed, test_src)
        else:
            for role, hf in [("validation",val_src),("test",test_src)]:
                if hf not in raw: continue
                ds = raw[hf]; n = n_va if role=="validation" else n_te
                if n and n < len(ds):
                    idx = rng.choice(len(ds),size=n,replace=False).tolist(); idx.sort()
                    ds = ds.select(idx)
                    if role=="test": _save_test_indices(idx, cfg, seed, hf)
                result[role] = ds
        tr_src = srcs.get("train","train")
        if tr_src in raw:
            ds = raw[tr_src]
            if n_tr and n_tr < len(ds):
                idx = rng.choice(len(ds),size=n_tr,replace=False).tolist(); idx.sort()
                ds = ds.select(idx)
            result["train"] = ds
        return DatasetDict(result)
    def _save_test_indices(indices, cfg, seed, hf_split="unknown"):
        out = Path(cfg.get("training",{}).get("output_dir","experiments/default"))
        out.mkdir(parents=True, exist_ok=True)
        (out/"test_indices.json").write_text(
            json.dumps({"seed":seed,"dataset":cfg["dataset"].get("name"),
                        "hf_source_split":hf_split,"num_test_examples":len(indices),
                        "indices":indices}, indent=2), encoding="utf-8")
""")

# ── src/evaluation/metrics.py ──────────────────────────────────────────────────
write("src/evaluation/metrics.py", """
    from __future__ import annotations
    import re, string, logging
    from collections import Counter
    import numpy as np
    logger = logging.getLogger(__name__)
    def normalize_answer(s):
        def rm_art(t): return re.sub(r"\\b(a|an|the)\\b"," ",t)
        def ws(t): return " ".join(t.split())
        def rm_punc(t): return "".join(c for c in t if c not in set(string.punctuation))
        return ws(rm_art(rm_punc(s.lower())))
    def get_tokens(s): return normalize_answer(s).split()
    def compute_exact_match(p,g): return float(normalize_answer(p)==normalize_answer(g))
    def compute_f1(p,g):
        pt=get_tokens(p); gt=get_tokens(g)
        common=Counter(pt)&Counter(gt); n=sum(common.values())
        if n==0: return 0.0
        pr=n/len(pt); rc=n/len(gt)
        return 2*pr*rc/(pr+rc)
    def compute_qa_metrics(predictions, references, report_unanswerable_breakdown=True):
        ems=[compute_exact_match(p,r) for p,r in zip(predictions,references)]
        f1s=[compute_f1(p,r) for p,r in zip(predictions,references)]
        res={"exact_match":round(np.mean(ems)*100,2),"f1":round(np.mean(f1s)*100,2),"num_examples":len(predictions)}
        if report_unanswerable_breakdown:
            ans=[(p,r,e,f) for p,r,e,f in zip(predictions,references,ems,f1s) if normalize_answer(r)!="unanswerable"]
            una=[(p,r,e,f) for p,r,e,f in zip(predictions,references,ems,f1s) if normalize_answer(r)=="unanswerable"]
            if ans:
                res["answerable_em"]=round(np.mean([x[2] for x in ans])*100,2)
                res["answerable_f1"]=round(np.mean([x[3] for x in ans])*100,2)
                res["answerable_n"]=len(ans)
            if una:
                res["unanswerable_em"]=round(np.mean([x[2] for x in una])*100,2)
                res["unanswerable_f1"]=round(np.mean([x[3] for x in una])*100,2)
                res["unanswerable_n"]=len(una)
        return res
    def compute_rouge(predictions, references):
        from rouge_score import rouge_scorer
        scorer = rouge_scorer.RougeScorer(["rouge1","rouge2","rougeL"],use_stemmer=True)
        sc={"rouge1":[],"rouge2":[],"rougeL":[]}
        for p,r in zip(predictions,references):
            res=scorer.score(r,p)
            for k in sc: sc[k].append(res[k].fmeasure)
        return {"rouge1":round(np.mean(sc["rouge1"])*100,2),
                "rouge2":round(np.mean(sc["rouge2"])*100,2),
                "rougeL":round(np.mean(sc["rougeL"])*100,2),
                "num_examples":len(predictions)}
    def make_compute_metrics_fn(task, tokenizer):
        def _decode(ids):
            ids = np.where(ids==-100, tokenizer.pad_token_id, ids)
            return tokenizer.batch_decode(ids, skip_special_tokens=True)
        def compute_metrics(eval_pred):
            preds, labels = eval_pred
            if isinstance(preds,tuple): preds=preds[0]
            if preds.ndim==3: preds=np.argmax(preds,axis=-1)
            dp=[p.strip() for p in _decode(preds)]
            dl=[l.strip() for l in _decode(labels)]
            if task=="qa": return compute_qa_metrics(dp,dl)
            if task=="summarization": return compute_rouge(dp,dl)
            return {}
        return compute_metrics
""")

# ── src/evaluation/evaluator.py ────────────────────────────────────────────────
write("src/evaluation/evaluator.py", """
    from __future__ import annotations
    import logging, time
    import numpy as np, torch
    from .metrics import compute_qa_metrics, compute_rouge
    logger = logging.getLogger(__name__)
    _TASK_FN = {"qa":compute_qa_metrics,"summarization":compute_rouge}
    def _split(ex):
        labels=ex["labels"]
        start=next(i for i,t in enumerate(labels) if t!=-100)
        return list(ex["input_ids"][:start]),[t for t in labels[start:] if t!=-100]
    def _postprocess(pred,task):
        pred=pred.split("\\n###")[0].strip()
        if task=="qa": pred=pred.split("\\n")[0].strip()
        return pred
    def evaluate_model(model, tokenizer, dataset, cfg, batch_size=8):
        task=cfg["task"]; max_new=cfg["dataset"].get("max_target_length",64)
        device=next(model.parameters()).device; pad=tokenizer.pad_token_id
        model.eval()
        pairs=[_split(dataset[i]) for i in range(len(dataset))]
        all_preds=[]; all_refs=[tokenizer.decode(r,skip_special_tokens=True).strip() for _,r in pairs]
        gen_s=0.0
        with torch.no_grad():
            for b in range(0,len(pairs),batch_size):
                chunk=pairs[b:b+batch_size]; prompts=[p for p,_ in chunk]
                w=max(len(p) for p in prompts)
                ids=torch.tensor([[pad]*(w-len(p))+p for p in prompts],device=device)
                msk=torch.tensor([[0]*(w-len(p))+[1]*len(p) for p in prompts],device=device)
                if device.type=="cuda": torch.cuda.synchronize()
                t0=time.perf_counter()
                out=model.generate(input_ids=ids,attention_mask=msk,max_new_tokens=max_new,
                    do_sample=False,use_cache=True,pad_token_id=pad,eos_token_id=tokenizer.eos_token_id)
                if device.type=="cuda": torch.cuda.synchronize()
                gen_s+=time.perf_counter()-t0
                dec=tokenizer.batch_decode(out[:,w:],skip_special_tokens=True)
                all_preds.extend(_postprocess(p,task) for p in dec)
        fn=_TASK_FN.get(task)
        if fn is None: return {}
        metrics=fn(all_preds,all_refs)
        metrics["avg_latency_ms"]=round(gen_s*1000/len(all_preds),1)
        metrics["throughput_samples_per_s"]=round(len(all_preds)/gen_s,2)
        metrics["_predictions"]=all_preds; metrics["_references"]=all_refs
        return metrics
""")

# ── src/training/model_loader.py ───────────────────────────────────────────────
write("src/training/model_loader.py", """
    from __future__ import annotations
    import logging
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    logger = logging.getLogger(__name__)
    def load_tokenizer(cfg):
        path = cfg.get("tokenizer_name_or_path") or cfg["model_name_or_path"]
        tok = AutoTokenizer.from_pretrained(path, trust_remote_code=True, padding_side="right")
        if tok.pad_token is None: tok.pad_token = tok.eos_token
        return tok
    def load_base_model(cfg):
        path=cfg["model_name_or_path"]; bits=cfg.get("load_in_bits")
        bnb = _bnb(bits) if bits in (4,8) else None
        kw={"trust_remote_code":True,"device_map":"auto"}
        if bnb: kw["quantization_config"]=bnb
        elif torch.cuda.is_available():
            kw["torch_dtype"]=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
        else: kw["torch_dtype"]=torch.float32
        model=AutoModelForCausalLM.from_pretrained(path,**kw)
        model.config.use_cache=False; return model
    def _bnb(bits):
        if bits==4: return BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",bnb_4bit_compute_dtype=torch.bfloat16)
        return BitsAndBytesConfig(load_in_8bit=True)
    def attach_lora(model, cfg):
        from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
        pc=cfg.get("peft",{}); bits=cfg.get("load_in_bits")
        if bits in (4,8): model=prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
        lc=LoraConfig(r=pc.get("r",8),lora_alpha=pc.get("lora_alpha",16),
            lora_dropout=pc.get("lora_dropout",0.05),target_modules=pc.get("target_modules"),
            bias=pc.get("bias","none"),task_type=pc.get("task_type","CAUSAL_LM"))
        model=get_peft_model(model,lc)
        tr=sum(p.numel() for p in model.parameters() if p.requires_grad)
        tot=sum(p.numel() for p in model.parameters())
        pct=100*tr/tot if tot else 0
        print(f"\\n✅ LoRA attached: {tr:,} trainable / {tot:,} total ({pct:.3f}%)")
        cfg["_trainable_params"]=tr; cfg["_total_params"]=tot; cfg["_trainable_pct"]=round(pct,4)
        return model
    def load_peft_checkpoint(base_cfg, checkpoint_path):
        from peft import PeftModel
        base=load_base_model(base_cfg)
        model=PeftModel.from_pretrained(base,checkpoint_path)
        model.eval(); return model
""")

# ── src/training/trainer.py ────────────────────────────────────────────────────
write("src/training/trainer.py", """
    from __future__ import annotations
    import logging, time
    from pathlib import Path
    import torch
    from transformers import TrainingArguments, Trainer
    from .model_loader import load_tokenizer, load_base_model, attach_lora
    from ..data.dataset_loader import load_dataset_for_task
    from ..data.preprocessor import get_data_collator
    from ..evaluation.metrics import make_compute_metrics_fn
    from ..utils.experiment_tracker import ExperimentTracker
    from ..utils.config_loader import load_config
    logger = logging.getLogger(__name__)
    def _peak():
        return torch.cuda.max_memory_allocated()/(1024**2) if torch.cuda.is_available() else 0.0
    def run_training(config_path):
        cfg=load_config(config_path)
        out=Path(cfg["training"]["output_dir"]); out.mkdir(parents=True,exist_ok=True)
        tok=load_tokenizer(cfg)
        ds=load_dataset_for_task(cfg,tok)
        model=attach_lora(load_base_model(cfg),cfg)
        tracker=ExperimentTracker(cfg["results_csv"]); mem0=_peak()
        tc=cfg["training"]
        args=TrainingArguments(
            output_dir=str(out),num_train_epochs=tc.get("num_train_epochs",3),
            per_device_train_batch_size=tc.get("per_device_train_batch_size",4),
            per_device_eval_batch_size=tc.get("per_device_eval_batch_size",8),
            gradient_accumulation_steps=tc.get("gradient_accumulation_steps",4),
            learning_rate=float(tc.get("learning_rate",2e-4)),
            weight_decay=float(tc.get("weight_decay",0.01)),
            warmup_ratio=float(tc.get("warmup_ratio",0.03)),
            lr_scheduler_type=tc.get("lr_scheduler_type","cosine"),
            fp16=tc.get("fp16",False),bf16=tc.get("bf16",False),
            logging_dir=str(out/"logs"),logging_steps=tc.get("logging_steps",50),
            eval_strategy="steps",eval_steps=tc.get("eval_steps",200),
            save_strategy="steps",save_steps=tc.get("save_steps",200),
            save_total_limit=tc.get("save_total_limit",2),
            load_best_model_at_end=tc.get("load_best_model_at_end",True),
            metric_for_best_model=tc.get("metric_for_best_model","eval_loss"),
            seed=tc.get("seed",42),report_to="tensorboard",dataloader_pin_memory=False)
        trainer=Trainer(model=model,args=args,train_dataset=ds.get("train"),
            eval_dataset=ds.get("validation"),tokenizer=tok,
            data_collator=get_data_collator(tok),
            compute_metrics=make_compute_metrics_fn(cfg["task"],tok))
        t0=time.time(); trainer.train(); train_t=time.time()-t0
        mem=_peak()-mem0
        test_m={}
        if "test" in ds: test_m=trainer.evaluate(eval_dataset=ds["test"])
        model.save_pretrained(str(out/"adapter")); tok.save_pretrained(str(out/"adapter"))
        row={"experiment":cfg.get("experiment_name",""),"model":cfg["model_name_or_path"],
             "model_variant":cfg.get("model_variant","base"),
             "dataset":cfg["dataset"].get("name",""),"task":cfg["task"],
             "method":"LoRA" if not cfg.get("load_in_bits") else "QLoRA",
             "seed":tc.get("seed",42),
             "train_samples":len(ds.get("train",[])),"val_samples":len(ds.get("validation",[])),
             "test_samples":len(ds.get("test",[])),
             "trainable_pct":cfg.get("_trainable_pct"),
             "train_time_s":round(train_t,1),"peak_gpu_mb":round(mem,1),
             "checkpoint":str(out/"adapter"),
             **{k:round(v,4) if isinstance(v,float) else v for k,v in test_m.items()}}
        tracker.log(row); return test_m
""")

# ── configs/pilot_squad_qa.yaml ────────────────────────────────────────────────
write("configs/pilot_squad_qa.yaml", """
    experiment_name: "squad_qa_lora_pilot"
    task: "qa"
    model_name_or_path: "Qwen/Qwen1.5-0.5B"
    tokenizer_name_or_path: null
    model_variant: "base"
    use_flash_attention: false
    load_in_bits: null
    peft:
      method: "lora"
      r: 8
      lora_alpha: 16
      lora_dropout: 0.05
      target_modules:
        - "q_proj"
        - "v_proj"
      bias: "none"
      task_type: "CAUSAL_LM"
    dataset:
      name: "rajpurkar/squad_v2"
      version: "default"
      subset: null
      text_column: "context"
      question_column: "question"
      answer_column: "answers"
      max_source_length: 512
      max_target_length: 64
      train_samples: 5000
      val_samples: 500
      test_samples: 500
    training:
      output_dir: "experiments/squad_qa"
      num_train_epochs: 3
      per_device_train_batch_size: 4
      per_device_eval_batch_size: 8
      gradient_accumulation_steps: 4
      learning_rate: 2.0e-4
      weight_decay: 0.01
      warmup_ratio: 0.03
      lr_scheduler_type: "cosine"
      fp16: false
      bf16: true
      logging_steps: 50
      eval_steps: 200
      save_steps: 200
      save_total_limit: 2
      load_best_model_at_end: true
      metric_for_best_model: "eval_f1"
      seed: 42
    evaluation:
      metrics:
        - "exact_match"
        - "f1"
      record_efficiency: true
    results_csv: "experiments/results.csv"
""")

# ── configs/pilot_xlsum_summarization.yaml ─────────────────────────────────────
write("configs/pilot_xlsum_summarization.yaml", """
    experiment_name: "xlsum_summarization_lora_pilot"
    task: "summarization"
    model_name_or_path: "Qwen/Qwen1.5-0.5B"
    tokenizer_name_or_path: null
    model_variant: "base"
    use_flash_attention: false
    load_in_bits: null
    peft:
      method: "lora"
      r: 8
      lora_alpha: 16
      lora_dropout: 0.05
      target_modules:
        - "q_proj"
        - "v_proj"
      bias: "none"
      task_type: "CAUSAL_LM"
    dataset:
      name: "csebuetnlp/xlsum"
      version: "default"
      subset: "english"
      text_column: "text"
      summary_column: "summary"
      max_source_length: 512
      max_target_length: 84
      train_samples: 5000
      val_samples: 500
      test_samples: 500
    training:
      output_dir: "experiments/xlsum_summarization"
      num_train_epochs: 3
      per_device_train_batch_size: 4
      per_device_eval_batch_size: 8
      gradient_accumulation_steps: 4
      learning_rate: 2.0e-4
      weight_decay: 0.01
      warmup_ratio: 0.03
      lr_scheduler_type: "cosine"
      fp16: false
      bf16: true
      logging_steps: 50
      eval_steps: 200
      save_steps: 200
      save_total_limit: 2
      load_best_model_at_end: true
      metric_for_best_model: "eval_rougeL"
      seed: 42
    evaluation:
      metrics:
        - "rouge1"
        - "rouge2"
        - "rougeL"
      record_efficiency: true
    results_csv: "experiments/results.csv"
""")

# ── configs/llm_baseline_qwen7b.yaml ──────────────────────────────────────────
write("configs/llm_baseline_qwen7b.yaml", """
    experiment_name: "squad_qa_llm_baseline"
    task: "qa"
    model_name_or_path: "Qwen/Qwen1.5-7B"
    tokenizer_name_or_path: null
    model_variant: "base"
    use_flash_attention: false
    load_in_bits: 4
    peft:
      method: "lora"
      r: 8
      lora_alpha: 16
      lora_dropout: 0.05
      target_modules:
        - "q_proj"
        - "v_proj"
      bias: "none"
      task_type: "CAUSAL_LM"
    dataset:
      name: "rajpurkar/squad_v2"
      version: "default"
      subset: null
      text_column: "context"
      question_column: "question"
      answer_column: "answers"
      max_source_length: 512
      max_target_length: 64
      train_samples: 5000
      val_samples: 500
      test_samples: 500
    training:
      output_dir: "experiments/llm_baseline"
      num_train_epochs: 3
      per_device_train_batch_size: 2
      per_device_eval_batch_size: 4
      gradient_accumulation_steps: 8
      learning_rate: 2.0e-4
      weight_decay: 0.01
      warmup_ratio: 0.03
      lr_scheduler_type: "cosine"
      fp16: false
      bf16: false
      logging_steps: 50
      eval_steps: 200
      save_steps: 200
      save_total_limit: 2
      load_best_model_at_end: false
      metric_for_best_model: "eval_f1"
      seed: 42
    evaluation:
      metrics:
        - "exact_match"
        - "f1"
      record_efficiency: true
    results_csv: "experiments/results.csv"
""")

# Done
print("✅ All source files written to", ROOT)
print("\nFiles created:")
for f in sorted(ROOT.rglob("*.py")): print(" ", f.relative_to(ROOT))
for f in sorted(ROOT.rglob("*.yaml")): print(" ", f.relative_to(ROOT))

In [ ]:
# Add project root to Python path so all imports work
import sys
PROJECT_ROOT = "/content/Mtech"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
import os
os.chdir(PROJECT_ROOT)
print(f"✅ Working dir: {os.getcwd()}")

# Quick import check
from src.utils.config_loader import load_config
from src.utils.experiment_tracker import ExperimentTracker
from src.data.prompts import build_qa_prompt, build_summarization_prompt
from src.evaluation.metrics import compute_qa_metrics, compute_rouge
print("✅ All project imports OK")

---
## 🧪 Step 3 — Smoke Test (9 checks)
> ✅ All checks must print `[OK]` — fix any `[FAIL]` before continuing.

In [ ]:
import traceback

PASS = "[OK] "; FAIL = "[FAIL]"; failures = []

def check(name, fn):
    try:
        r = fn(); print(f"{PASS} {name}"); return r
    except Exception as e:
        print(f"{FAIL} {name}: {e}")
        failures.append(name); traceback.print_exc(); return None

# [1] Config
def _c1():
    cfg = load_config("configs/pilot_squad_qa.yaml")
    assert cfg["task"] == "qa"
    assert cfg["model_name_or_path"] == "Qwen/Qwen1.5-0.5B"
    assert cfg["training"]["seed"] == 42
    return cfg
cfg = check("[1] Config loads correctly", _c1)
if cfg is None:
    raise RuntimeError("Cannot continue without config")

# [2] Dataset
def _c2():
    from datasets import load_dataset
    from src.data.dataset_loader import _subsample
    c = dict(cfg); c["dataset"] = dict(cfg["dataset"])
    c["dataset"]["train_samples"]=50; c["dataset"]["val_samples"]=25; c["dataset"]["test_samples"]=25
    c["training"] = dict(cfg["training"]); c["training"]["output_dir"]="experiments/smoke_test"
    raw = load_dataset("rajpurkar/squad_v2")
    r = _subsample(raw, c["dataset"], c)
    assert len(r["train"]) == 50
    assert len(r["validation"]) == 25
    assert len(r["test"]) == 25
    un = [e for e in list(r["test"]) if not e["answers"]["text"]]
    print(f"         test: {25-len(un)} answerable, {len(un)} unanswerable")
    assert len(un) > 0, "No unanswerable examples in test split"
check("[2] Dataset splits: correct sizes, unanswerable present", _c2)

# [3] Prompt format
def _c3():
    p = build_qa_prompt("Paris is the capital of France.", "What is the capital?")
    assert "### Context:" in p and "### Question:" in p and "### Answer:" in p
    assert "unanswerable" in build_qa_prompt("ctx","q") + "unanswerable"
    print(f"         Sample: {p[:60].strip()}...")
check("[3] Prompt format correct", _c3)

# [4,5,6] Tokenizer + masking
def _c4():
    from transformers import AutoTokenizer
    from src.data.preprocessor import _tokenize_prompt_completion, _IGNORE_INDEX
    tok = AutoTokenizer.from_pretrained("Qwen/Qwen1.5-0.5B", trust_remote_code=True)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    prompts = [
        "### Context:\nParis is the capital.\n\n### Question:\nCapital?\n\n### Answer:\nParis",
        "### Context:\nLoRA is PEFT.\n\n### Question:\nWhat is LoRA?\n\n### Answer:\nunanswerable",
    ]
    r = _tokenize_prompt_completion({"prompt": prompts}, tok, 256)
    assert len(r["input_ids"]) == 2
    for i, (ids, labels) in enumerate(zip(r["input_ids"], r["labels"])):
        assert len(ids) == len(labels)
        comp = [t for t in labels if t != _IGNORE_INDEX]
        assert len(comp) > 0
        print(f"         Ex {i}: completion_tokens={len(comp)}, masked={(labels.count(_IGNORE_INDEX))}")
check("[4,5,6] Tokenisation + masking: prompt=-100, completion≠-100", _c4)

# [7] Base model inference
def _c7():
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    print("         Loading Qwen1.5-0.5B (~30s)...")
    tok = AutoTokenizer.from_pretrained("Qwen/Qwen1.5-0.5B", trust_remote_code=True)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen1.5-0.5B", trust_remote_code=True,
        torch_dtype=torch.bfloat16, device_map="auto")
    model.eval()
    prompt = "### Context:\nParis is the capital of France.\n\n### Question:\nCapital?\n\n### Answer:\n"
    inp = tok(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=20, do_sample=False, pad_token_id=tok.pad_token_id)
    pred = tok.decode(out[0, inp["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    print(f"         Prediction: '{pred[:60]}'")
    assert isinstance(pred, str)
    del model; torch.cuda.empty_cache()
check("[7] Base model inference runs, returns string", _c7)

# [8] Metrics
def _c8():
    assert compute_exact_match := __import__("src.evaluation.metrics", fromlist=["compute_exact_match"]).compute_exact_match
    assert compute_exact_match("Paris","Paris") == 1.0
    assert compute_exact_match("paris","Paris") == 1.0
    assert compute_exact_match("London","Paris") == 0.0
    r = compute_qa_metrics(["Paris","unanswerable","London","unanswerable"],
                            ["Paris","unanswerable","Paris","unanswerable"])
    assert r["answerable_em"] == 50.0
    assert r["unanswerable_em"] == 100.0
    rouge = compute_rouge(["A new fish was found."], ["Scientists discovered a new fish."])
    assert rouge["rouge1"] > 0
    print(f"         EM={r['exact_match']} | Ans EM={r['answerable_em']} | UnAns EM={r['unanswerable_em']}")
check("[8] Metrics: EM, F1, ROUGE, unanswerable breakdown", _c8)

# [9] Experiment tracker
def _c9():
    t = ExperimentTracker("experiments/smoke_test/results.csv")
    t.log({"experiment":"smoke_test","model":"Qwen/Qwen1.5-0.5B","task":"qa",
           "method":"base_zero_shot","exact_match":20.0,"f1":33.5,
           "answerable_em":25.0,"unanswerable_em":15.0})
    df = t.load()
    assert len(df) >= 1 and "exact_match" in df.columns
check("[9] ExperimentTracker writes and reads CSV", _c9)

print()
print("=" * 60)
if failures:
    print(f"❌ SMOKE TEST FAILED — {len(failures)} check(s) failed: {failures}")
else:
    print("✅ ALL SMOKE TEST CHECKS PASSED — pipeline is verified")
    print("   Next: run the Quick Demo or jump straight to the experiments.")

---
## ⚡ Step 4 — Quick 3-Way Demo (5 built-in examples, ~10 min)
Runs the full pipeline on hard-coded examples so you can see the comparison table before investing 2+ hours.

In [ ]:
import torch, time, logging
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
from src.evaluation.metrics import compute_qa_metrics

logging.basicConfig(level=logging.WARNING)  # suppress verbose HF logs
device = "cuda" if torch.cuda.is_available() else "cpu"

QA_EXAMPLES = [
    {"context":"The Eiffel Tower is in Paris, France. It was built in 1889.",
     "question":"Where is the Eiffel Tower?", "answer":"Paris, France"},
    {"context":"Python was created by Guido van Rossum and first released in 1991.",
     "question":"Who created Python?", "answer":"Guido van Rossum"},
    {"context":"LoRA adds trainable low-rank matrices to frozen transformer layers.",
     "question":"What does LoRA add to transformers?", "answer":"trainable low-rank matrices"},
    {"context":"The Amazon River flows through Brazil, Peru, and Colombia.",
     "question":"Which countries does the Amazon flow through?", "answer":"Brazil, Peru, and Colombia"},
    {"context":"BERT stands for Bidirectional Encoder Representations from Transformers.",
     "question":"What does BERT stand for?", "answer":"Bidirectional Encoder Representations from Transformers"},
]

def build_prompt(ctx, q):
    return f"### Context:\n{ctx}\n\n### Question:\n{q}\n\n### Answer:\n"

def infer(model, tok, prompts, max_new=40):
    model.eval(); preds = []; t0 = time.perf_counter()
    for p in prompts:
        inp = tok(p, return_tensors="pt").to(device)
        with torch.no_grad():
            out = model.generate(**inp, max_new_tokens=max_new, do_sample=False,
                                  pad_token_id=tok.pad_token_id, eos_token_id=tok.eos_token_id)
        pred = tok.decode(out[0, inp["input_ids"].shape[1]:], skip_special_tokens=True)
        preds.append(pred.split("\n")[0].strip())
    return preds, (time.perf_counter()-t0)*1000

prompts = [build_prompt(e["context"],e["question"]) for e in QA_EXAMPLES]
answers = [e["answer"] for e in QA_EXAMPLES]
full_texts = [p+a for p,a in zip(prompts, answers)]

# ── Load SLM ──────────────────────────────────────────────────────────────────
print("Loading Qwen1.5-0.5B...")
tok = AutoTokenizer.from_pretrained("Qwen/Qwen1.5-0.5B", trust_remote_code=True)
if tok.pad_token is None: tok.pad_token = tok.eos_token
base = AutoModelForCausalLM.from_pretrained("Qwen/Qwen1.5-0.5B", trust_remote_code=True,
    torch_dtype=torch.bfloat16, device_map=device)
base.config.use_cache = False

# ── Base SLM inference ────────────────────────────────────────────────────────
print("\nStep 1/3: Base SLM zero-shot inference...")
base_preds, base_lat = infer(base, tok, prompts)
base_m = compute_qa_metrics(base_preds, answers)
print(f"  EM={base_m['exact_match']}  F1={base_m['f1']}  Lat={base_lat/len(prompts):.0f}ms/ex")

# ── Attach LoRA + fine-tune ───────────────────────────────────────────────────
print("\nStep 2/3: Attach LoRA and fine-tune (10 steps)...")
import torch.optim as optim
lora_cfg = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05,
    target_modules=["q_proj","v_proj"], bias="none", task_type="CAUSAL_LM")
peft_model = get_peft_model(base, lora_cfg)
tr = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
tot = sum(p.numel() for p in peft_model.parameters())
print(f"  LoRA: {tr:,} trainable / {tot:,} total ({100*tr/tot:.3f}%)")

opt = optim.AdamW(peft_model.parameters(), lr=2e-4)
peft_model.train()
for step in range(10):
    total_loss = 0.0
    for text in full_texts:
        inp = tok(text, return_tensors="pt", truncation=True, max_length=256).to(device)
        labels = inp["input_ids"].clone()
        out = peft_model(**inp, labels=labels)
        out.loss.backward(); total_loss += out.loss.item()
    opt.step(); opt.zero_grad()
    if (step+1) % 5 == 0:
        print(f"  step {step+1}/10 | avg loss: {total_loss/len(full_texts):.4f}")

# ── PEFT-SLM inference ────────────────────────────────────────────────────────
print("\nStep 3/3: PEFT-SLM inference...")
peft_preds, peft_lat = infer(peft_model, tok, prompts)
peft_m = compute_qa_metrics(peft_preds, answers)
print(f"  EM={peft_m['exact_match']}  F1={peft_m['f1']}  Lat={peft_lat/len(prompts):.0f}ms/ex")

# ── 3-way table ───────────────────────────────────────────────────────────────
print("\n" + "="*65)
print("  QUICK DEMO — 3-WAY COMPARISON (5 examples, 10 LoRA steps)")
print("="*65)
print(f"  {'Model/Method':<35} {'EM':>6} {'F1':>6} {'Lat(ms/ex)':>12}")
print("-"*65)
print(f"  {'SLM base (zero-shot)':<35} {base_m['exact_match']:>6.1f} {base_m['f1']:>6.1f} {base_lat/len(prompts):>12.0f}")
print(f"  {'SLM + LoRA (PEFT, 10 steps) ← OURS':<35} {peft_m['exact_match']:>6.1f} {peft_m['f1']:>6.1f} {peft_lat/len(prompts):>12.0f}")
print("="*65)
print("  (LLM baseline skipped in demo — run Step 5C for full experiment)")

print("\nPer-example predictions:")
for i, ex in enumerate(QA_EXAMPLES):
    print(f"  [{i+1}] Ref  : {ex['answer']}")
    print(f"       Base : {base_preds[i][:80]}")
    print(f"       PEFT : {peft_preds[i][:80]}")

del base, peft_model; torch.cuda.empty_cache()
print("\n✅ Quick demo complete — GPU memory freed")

---
## 📊 Step 5 — Full SQuAD v2 QA Experiments (Phase 1B)
Run **5A → 5B → 5C → 5D** in order. Each step logs results to `experiments/results.csv`.

In [ ]:
# ── 5A  Base SLM zero-shot baseline (~15-20 min) ─────────────────────────────
import sys, os
sys.path.insert(0, "/content/Mtech")
os.chdir("/content/Mtech")

from src.utils.config_loader import load_config
from src.utils.logging_setup import setup_logging
from src.training.model_loader import load_tokenizer, load_base_model
from src.data.dataset_loader import load_dataset_for_task
from src.evaluation.evaluator import evaluate_model
from src.utils.experiment_tracker import ExperimentTracker

setup_logging("INFO")
print("=" * 60)
print("5A — Base SLM zero-shot baseline on SQuAD v2")
print("Expected: ~15-20 minutes on T4")
print("=" * 60)

cfg = load_config("configs/pilot_squad_qa.yaml")
tok = load_tokenizer(cfg)
model = load_base_model(cfg)
model.eval()

ds = load_dataset_for_task(cfg, tok)
print(f"\nTest split: {len(ds['test'])} examples")

metrics = evaluate_model(model, tok, ds["test"], cfg)
print("\n📊 Base SLM Results:")
for k,v in metrics.items():
    if not k.startswith("_"): print(f"  {k}: {v}")

ExperimentTracker(cfg["results_csv"]).log({
    "experiment": cfg["experiment_name"]+"_base",
    "model": cfg["model_name_or_path"], "model_variant": "base",
    "dataset": cfg["dataset"]["name"], "task": cfg["task"],
    "method": "base_zero_shot", "seed": 42,
    "test_samples": len(ds["test"]),
    **{k:v for k,v in metrics.items() if not k.startswith("_")},
})

import torch
del model; torch.cuda.empty_cache()
print("\n✅ Step 5A complete — base SLM results logged")

In [ ]:
# ── 5B  LoRA fine-tuning — YOUR RESEARCH CONTRIBUTION (~90 min) ──────────────
# Watch: loss should decrease at each logging step.
# If you get OOM: run the OOM fix cell below, then re-run this cell.

print("=" * 60)
print("5B — LoRA fine-tuning on SQuAD v2 (YOUR result)")
print("Expected: ~90 min on T4 (3 epochs × 5000 examples)")
print("Watch: loss should decrease — best checkpoint saved to experiments/squad_qa/adapter/")
print("=" * 60)

from src.training.trainer import run_training
metrics = run_training("configs/pilot_squad_qa.yaml")

print("\n📊 LoRA Fine-tuned Results:")
for k,v in metrics.items():
    if not k.startswith("_"): print(f"  {k}: {v}")
print("\n✅ Step 5B complete")

In [ ]:
# ── 5B OOM fix — ONLY run if you got an out-of-memory error above ─────────────
import yaml, pathlib

cfg_path = pathlib.Path("/content/Mtech/configs/pilot_squad_qa.yaml")
cfg_data = yaml.safe_load(cfg_path.read_text())
cfg_data["load_in_bits"] = 4
cfg_path.write_text(yaml.dump(cfg_data, default_flow_style=False))
print("✅ Switched to QLoRA (4-bit) — re-run Step 5B cell now")

In [ ]:
# ── 5C  LLM baseline — Qwen1.5-7B zero-shot (~25 min) ───────────────────────
# The config already has load_in_bits: 4 for safety on T4.

print("=" * 60)
print("5C — LLM baseline (Qwen1.5-7B, 4-bit, zero-shot)")
print("Expected: ~25 min on T4")
print("=" * 60)

from src.utils.config_loader import load_config
from src.training.model_loader import load_tokenizer, load_base_model
from src.data.dataset_loader import load_dataset_for_task
from src.evaluation.evaluator import evaluate_model
from src.utils.experiment_tracker import ExperimentTracker

cfg7 = load_config("configs/llm_baseline_qwen7b.yaml")
tok7 = load_tokenizer(cfg7)
model7 = load_base_model(cfg7)
model7.eval()

# Use same test split as 5A/5B (same seed + same dataset config)
ds7 = load_dataset_for_task(cfg7, tok7)
print(f"\nTest split: {len(ds7['test'])} examples")

m7 = evaluate_model(model7, tok7, ds7["test"], cfg7)
print("\n📊 Qwen1.5-7B Results:")
for k,v in m7.items():
    if not k.startswith("_"): print(f"  {k}: {v}")

ExperimentTracker(cfg7["results_csv"]).log({
    "experiment": cfg7["experiment_name"]+"_llm_base",
    "model": cfg7["model_name_or_path"], "model_variant": "base",
    "dataset": cfg7["dataset"]["name"], "task": cfg7["task"],
    "method": "base_zero_shot", "seed": 42,
    "test_samples": len(ds7["test"]),
    **{k:v for k,v in m7.items() if not k.startswith("_")},
})

import torch
del model7; torch.cuda.empty_cache()
print("\n✅ Step 5C complete")

In [ ]:
# ── 5D  Compile 3-way comparison table ───────────────────────────────────────
import pandas as pd, matplotlib.pyplot as plt
from src.utils.experiment_tracker import ExperimentTracker

print("=" * 70)
print("3-WAY COMPARISON — SQuAD v2 QA — 500 test examples — seed=42")
print("=" * 70)
ExperimentTracker("experiments/results.csv").print_summary()

# Bar chart
try:
    df = pd.read_csv("experiments/results.csv")
    qa = df[df["task"]=="qa"].dropna(subset=["exact_match","f1"])
    if len(qa) >= 2:
        fig, axes = plt.subplots(1,2, figsize=(11,4))
        fig.suptitle("SQuAD v2 — 3-Way Comparison", fontweight="bold")
        colors = ["#6b7280","#3b82f6","#7c3aed"]
        labels = qa["method"].tolist()
        for ax, metric, title in zip(axes, ["exact_match","f1"], ["Exact Match (%)","F1 (%)"]):
            ax.bar(labels, qa[metric].tolist(), color=colors[:len(labels)])
            ax.set_title(title); ax.set_ylabel("%")
            ax.tick_params(axis="x", rotation=15)
        plt.tight_layout()
        plt.savefig("experiments/qa_results.png", dpi=150, bbox_inches="tight")
        plt.show()
        print("\n✅ Chart saved to experiments/qa_results.png")
except Exception as e:
    print(f"Chart skipped: {e}")

---
## 📝 Step 6 — XL-Sum Summarization Experiments (Phase 1C)
Run after Step 5 is complete.

In [ ]:
# ── 6A  Base SLM summarization baseline (~15-20 min) ─────────────────────────
print("=" * 60)
print("6A — Base SLM zero-shot baseline on XL-Sum")
print("Expected: ~15-20 min on T4")
print("=" * 60)

from src.utils.config_loader import load_config
from src.training.model_loader import load_tokenizer, load_base_model
from src.data.dataset_loader import load_dataset_for_task
from src.evaluation.evaluator import evaluate_model
from src.utils.experiment_tracker import ExperimentTracker

cfg_s = load_config("configs/pilot_xlsum_summarization.yaml")
tok_s = load_tokenizer(cfg_s)
model_s = load_base_model(cfg_s)
model_s.eval()

ds_s = load_dataset_for_task(cfg_s, tok_s)
print(f"\nTest split: {len(ds_s['test'])} examples")

m_s = evaluate_model(model_s, tok_s, ds_s["test"], cfg_s)
print("\n📊 Base SLM Summarization Results:")
for k,v in m_s.items():
    if not k.startswith("_"): print(f"  {k}: {v}")

ExperimentTracker(cfg_s["results_csv"]).log({
    "experiment": cfg_s["experiment_name"]+"_base",
    "model": cfg_s["model_name_or_path"], "model_variant": "base",
    "dataset": cfg_s["dataset"]["name"], "task": cfg_s["task"],
    "method": "base_zero_shot", "seed": 42,
    "test_samples": len(ds_s["test"]),
    **{k:v for k,v in m_s.items() if not k.startswith("_")},
})

import torch
del model_s; torch.cuda.empty_cache()
print("\n✅ Step 6A complete")

In [ ]:
# ── 6B  LoRA fine-tuning on XL-Sum (~2 hours) ────────────────────────────────
print("=" * 60)
print("6B — LoRA fine-tuning on XL-Sum summarization")
print("Expected: ~2 hours on T4 (3 epochs × 5000 examples)")
print("=" * 60)

from src.training.trainer import run_training
m_lora_s = run_training("configs/pilot_xlsum_summarization.yaml")

print("\n📊 XL-Sum LoRA Results:")
for k,v in m_lora_s.items():
    if not k.startswith("_"): print(f"  {k}: {v}")
print("\n✅ Step 6B complete")

---
## 📈 Step 7 — View All Results & Download

In [ ]:
# ── 7.1  Print full results table ────────────────────────────────────────────
import pandas as pd
from src.utils.experiment_tracker import ExperimentTracker

print("="*70)
print("FULL EXPERIMENT RESULTS")
print("="*70)
ExperimentTracker("experiments/results.csv").print_summary()

try:
    df = pd.read_csv("experiments/results.csv")
    print(f"\nTotal runs logged: {len(df)}")
    display(df)
except: pass

In [ ]:
# ── 7.2  Download results.csv to your computer ───────────────────────────────
import os
from google.colab import files

if os.path.exists("experiments/results.csv"):
    files.download("experiments/results.csv")
    print("✅ Downloading results.csv")
else:
    print("❌ No results.csv yet — run the experiment cells first")

In [ ]:
# ── 7.3  Zip and download the LoRA adapter (optional) ───────────────────────
import os
from google.colab import files

adapter_path = "experiments/squad_qa/adapter"
if os.path.exists(adapter_path):
    !cd /content/Mtech && zip -r squad_qa_adapter.zip experiments/squad_qa/adapter/
    files.download("/content/Mtech/squad_qa_adapter.zip")
    print("✅ Downloading squad_qa_adapter.zip")
else:
    print("❌ Adapter not found — run Step 5B first")

---
## 💡 Troubleshooting

| Problem | Fix |
|---|---|
| No GPU / CUDA not available | Runtime → Change runtime type → T4 GPU |
| OOM during 0.5B training | Run the **5B OOM fix** cell (switches to 4-bit QLoRA) |
| OOM during 7B inference | Already handled — `llm_baseline_qwen7b.yaml` has `load_in_bits: 4` |
| `ModuleNotFoundError` | Re-run the **Step 2** cell (write source files), then the sys.path cell |
| Session disconnects | Results in `experiments/results.csv` — download it with Step 7.2 before session ends |
| Slow training | Normal — T4 takes ~90 min for 5000 examples × 3 epochs. Loss should decrease. |
| `eval_strategy` error | You may need `transformers>=4.45` — re-run Step 1 install cell |

### Expected results (approximate, base model)
```
SQuAD v2 QA:
  Base SLM (0.5B zero-shot) : EM ~5-15%,  F1 ~15-25%
  LoRA fine-tuned (ours)    : EM ~40-55%,  F1 ~55-70%  ← improvement
  LLM (7B zero-shot)        : EM ~20-35%,  F1 ~35-50%

XL-Sum Summarization:
  Base SLM (zero-shot)      : R1 ~5-15%
  LoRA fine-tuned (ours)    : R1 ~20-35%  ← improvement
```